In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/behrad3d/nasa-cmaps/CMaps/RUL_FD002.txt
/kaggle/input/datasets/behrad3d/nasa-cmaps/CMaps/test_FD003.txt
/kaggle/input/datasets/behrad3d/nasa-cmaps/CMaps/Damage Propagation Modeling.pdf
/kaggle/input/datasets/behrad3d/nasa-cmaps/CMaps/readme.txt
/kaggle/input/datasets/behrad3d/nasa-cmaps/CMaps/train_FD003.txt
/kaggle/input/datasets/behrad3d/nasa-cmaps/CMaps/test_FD004.txt
/kaggle/input/datasets/behrad3d/nasa-cmaps/CMaps/train_FD004.txt
/kaggle/input/datasets/behrad3d/nasa-cmaps/CMaps/x.txt
/kaggle/input/datasets/behrad3d/nasa-cmaps/CMaps/test_FD002.txt
/kaggle/input/datasets/behrad3d/nasa-cmaps/CMaps/train_FD001.txt
/kaggle/input/datasets/behrad3d/nasa-cmaps/CMaps/train_FD002.txt
/kaggle/input/datasets/behrad3d/nasa-cmaps/CMaps/RUL_FD001.txt
/kaggle/input/datasets/behrad3d/nasa-cmaps/CMaps/RUL_FD004.txt
/kaggle/input/datasets/behrad3d/nasa-cmaps/CMaps/RUL_FD003.txt
/kaggle/input/datasets/behrad3d/nasa-cmaps/CMaps/test_FD001.txt
/kaggle/input/datasets/behrad3d/nasa

# Data Preparation

In [2]:


import os
import numpy as np
import pandas as pd

# 1. Paths


DATA_PATH = "/kaggle/input/datasets/behrad3d/nasa-cmaps/CMaps"

files = {
    "FD001": "train_FD001.txt",
    "FD002": "train_FD002.txt",
    "FD003": "train_FD003.txt",
    "FD004": "train_FD004.txt",
}



# 2. Column Names


columns = [
    "engine_id",
    "cycle",
    "op_setting_1",
    "op_setting_2",
    "op_setting_3",
]

for i in range(1, 22):
    columns.append(f"sensor_{i}")



# 3. Load FD001-FD004


all_data = []

for fd_name, file_name in files.items():

    path = os.path.join(DATA_PATH, file_name)

    df = pd.read_csv(
        path,
        sep=r"\s+",
        header=None,
        names=columns,
    )

    # Keep original dataset source
    df["source_fd"] = fd_name

    # Important: engine_id repeats across FD files.
    # This creates a unique aircraft ID across all files.
    df["global_engine_id"] = (
        df["source_fd"] + "_engine_" + df["engine_id"].astype(str)
    )

    all_data.append(df)


combined_df = pd.concat(
    all_data,
    ignore_index=True,
)

print("Combined shape:")
print(combined_df.shape)

print("\nSource distribution:")
print(combined_df["source_fd"].value_counts())



# 4. Calculate RUL


def add_rul(df):

    max_cycle = (
        df.groupby("global_engine_id")["cycle"]
        .max()
        .reset_index()
        .rename(columns={"cycle": "max_cycle"})
    )

    df = df.merge(
        max_cycle,
        on="global_engine_id",
        how="left",
    )

    df["RUL"] = df["max_cycle"] - df["cycle"]

    df = df.drop(columns=["max_cycle"])

    return df


combined_df = add_rul(combined_df)



# 5. Create Mixed Non-IID Airlines


NUM_AIRLINES = 4
SEED = 42

airline_names = [
    f"Airline_{i+1}"
    for i in range(NUM_AIRLINES)
]

# Different FD proportions for different airlines.
# This makes airlines non-IID while still mixing FD001-FD004.
mixing_weights = {
    "FD001": [0.45, 0.25, 0.15, 0.15],
    "FD002": [0.15, 0.45, 0.25, 0.15],
    "FD003": [0.15, 0.15, 0.45, 0.25],
    "FD004": [0.25, 0.15, 0.15, 0.45],
}


airline_engine_map = {
    airline: []
    for airline in airline_names
}


for source_fd, source_df in combined_df.groupby("source_fd"):

    engines = source_df["global_engine_id"].unique()

    rng = np.random.default_rng(SEED)
    rng.shuffle(engines)

    weights = np.array(mixing_weights[source_fd])
    weights = weights / weights.sum()

    split_sizes = np.floor(weights * len(engines)).astype(int)

    # Add leftover engines to the last airline
    split_sizes[-1] += len(engines) - split_sizes.sum()

    start = 0

    for airline, size in zip(airline_names, split_sizes):

        selected_engines = engines[start:start + size]

        airline_engine_map[airline].extend(selected_engines)

        start += size



# 6. Build Final Airline Dataset


airline_data = []

for airline, engines in airline_engine_map.items():

    temp = combined_df[
        combined_df["global_engine_id"].isin(engines)
    ].copy()

    temp["airline_id"] = airline

    airline_data.append(temp)


airline_df = pd.concat(
    airline_data,
    ignore_index=True,
)



# 7. Sort Final Dataset


airline_df = airline_df.sort_values(
    by=["airline_id", "global_engine_id", "cycle"]
).reset_index(drop=True)



# 8. Check Distribution


print("\nAircraft distribution per airline and FD:")
print(
    airline_df.groupby(
        ["airline_id", "source_fd"]
    )["global_engine_id"]
    .nunique()
)


print("\nRows per airline:")
print(
    airline_df.groupby("airline_id")
    .size()
)


print("\nFinal dataset head:")
print(airline_df.head())


print("\nImportant columns preview:")
print(
    airline_df[
        [
            "global_engine_id",
            "engine_id",
            "cycle",
            "source_fd",
            "airline_id",
            "RUL",
        ]
    ].head(20)
)

Combined shape:
(160359, 28)

Source distribution:
source_fd
FD004    61249
FD002    53759
FD003    24720
FD001    20631
Name: count, dtype: int64

Aircraft distribution per airline and FD:
airline_id  source_fd
Airline_1   FD001         45
            FD002         39
            FD003         15
            FD004         62
Airline_2   FD001         25
            FD002        117
            FD003         15
            FD004         37
Airline_3   FD001         15
            FD002         65
            FD003         45
            FD004         37
Airline_4   FD001         15
            FD002         39
            FD003         25
            FD004        113
Name: global_engine_id, dtype: int64

Rows per airline:
airline_id
Airline_1    36298
Airline_2    41988
Airline_3    37100
Airline_4    44973
dtype: int64

Final dataset head:
   engine_id  cycle  op_setting_1  op_setting_2  op_setting_3  sensor_1  \
0          1      1       -0.0007       -0.0004         100.0    518.67 

# Different feature no for different airline

In [3]:


airline_feature_map = {
    "Airline_1": [
        "op_setting_1",
        "op_setting_2",
        "op_setting_3",
    ] + [f"sensor_{i}" for i in range(1, 13)],   # 15 features total

    "Airline_2": [
        "op_setting_1",
        "op_setting_2",
        "op_setting_3",
    ] + [f"sensor_{i}" for i in range(1, 16)],   # 18 features total

    "Airline_3": [
        "op_setting_1",
        "op_setting_2",
        "op_setting_3",
    ] + [f"sensor_{i}" for i in range(1, 19)],   # 21 features total

    "Airline_4": [
        "op_setting_1",
        "op_setting_2",
        "op_setting_3",
    ] + [f"sensor_{i}" for i in range(1, 22)],   # 24 features total
}



airline_clients = {}

for airline_id, feature_list in airline_feature_map.items():

    selected_columns = (
        [
            "global_engine_id",
            "engine_id",
            "cycle",
            "source_fd",
            "airline_id",
        ]
        + feature_list
        + ["RUL"]
    )

    airline_clients[airline_id] = airline_df[
        airline_df["airline_id"] == airline_id
    ][selected_columns].copy()

    print("\n", airline_id)
    print("Number of input features:", len(feature_list))
    print("Selected features:")
    print(feature_list)
    print("Shape:", airline_clients[airline_id].shape)


 Airline_1
Number of input features: 15
Selected features:
['op_setting_1', 'op_setting_2', 'op_setting_3', 'sensor_1', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_5', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_10', 'sensor_11', 'sensor_12']
Shape: (36298, 21)

 Airline_2
Number of input features: 18
Selected features:
['op_setting_1', 'op_setting_2', 'op_setting_3', 'sensor_1', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_5', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15']
Shape: (41988, 24)

 Airline_3
Number of input features: 21
Selected features:
['op_setting_1', 'op_setting_2', 'op_setting_3', 'sensor_1', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_5', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_18']
Shape: (37100, 27)

 Airline_4
Number of input features: 24
Selected features

# Sensor Adapter: Convert Different Airline Features
#     into Same 64-Dimensional Latent Space

In [4]:

import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler



class SensorAdapter(nn.Module):

    def __init__(self, input_dim, latent_dim=64):

        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, latent_dim),
            nn.ReLU()
        )

    def forward(self, x):

        return self.encoder(x)



# Create 64 Latent Features for Every Airline


latent_airline_clients = {}
airline_adapters = {}
airline_scalers = {}


for airline_id, df in airline_clients.items():

    print("\nProcessing:", airline_id)

    # Get selected feature names for this airline
    feature_columns = airline_feature_map[airline_id]

    # Scale each airline's available features
    scaler = StandardScaler()

    X_scaled = scaler.fit_transform(
        df[feature_columns].values
    )

    # Convert to tensor
    X_tensor = torch.tensor(
        X_scaled,
        dtype=torch.float32
    )

    # Create adapter according to this airline's feature count
    adapter = SensorAdapter(
        input_dim=len(feature_columns),
        latent_dim=64
    )

    adapter.eval()

    # Extract latent features
    with torch.no_grad():

        X_latent = adapter(X_tensor).numpy()

    # Create latent feature column names
    latent_columns = [
        f"latent_{i+1}"
        for i in range(64)
    ]

    latent_df = pd.DataFrame(
        X_latent,
        columns=latent_columns
    )

    # Keep important identity and target columns
    meta_df = df[
        [
            "global_engine_id",
            "engine_id",
            "cycle",
            "source_fd",
            "airline_id",
            "RUL"
        ]
    ].reset_index(drop=True)

    latent_df = pd.concat(
        [
            meta_df,
            latent_df
        ],
        axis=1
    )

    # Save everything
    latent_airline_clients[airline_id] = latent_df
    airline_adapters[airline_id] = adapter
    airline_scalers[airline_id] = scaler

    print("Original feature count:", len(feature_columns))
    print("Latent shape:", latent_df.shape)

    latent_features = [
    f"latent_{i+1}"
    for i in range(64)
    ]

    X = latent_df[latent_features].values

    #print(latent_df.shape)
    print("Feature number:", len(latent_features))
    print("Feature Shape Without MetaData:", latent_df[latent_features].shape)


Processing: Airline_1
Original feature count: 15
Latent shape: (36298, 70)
Feature number: 64
Feature Shape Without MetaData: (36298, 64)

Processing: Airline_2
Original feature count: 18
Latent shape: (41988, 70)
Feature number: 64
Feature Shape Without MetaData: (41988, 64)

Processing: Airline_3
Original feature count: 21
Latent shape: (37100, 70)
Feature number: 64
Feature Shape Without MetaData: (37100, 64)

Processing: Airline_4
Original feature count: 24
Latent shape: (44973, 70)
Feature number: 64
Feature Shape Without MetaData: (44973, 64)


# Some Library

In [5]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from copy import deepcopy
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score, average_precision_score

# Window Creation

In [6]:
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

latent_features = [
    f"latent_{i+1}"
    for i in range(64)
]

WINDOW = 30
FAULT_THRESHOLD = 50
BATCH_SIZE = 64

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

Device: cpu


# Create LSTM Sequence

In [7]:
def create_sequences(df, feature_cols, window=30, fault_threshold=50):

    X_seq = []
    y_rul = []
    y_fault = []
    meta = []

    df = df.sort_values(
        ["global_engine_id", "cycle"]
    ).reset_index(drop=True)

    for engine_id, engine_df in df.groupby("global_engine_id"):

        engine_df = engine_df.sort_values("cycle")

        X = engine_df[feature_cols].values
        y = engine_df["RUL"].values

        if len(engine_df) <= window:
            continue

        for i in range(len(engine_df) - window):

            X_seq.append(
                X[i:i + window]
            )

            y_rul.append(
                y[i + window]
            )

            y_fault.append(
                float(y[i + window] <= fault_threshold)
            )

            meta.append({
                "global_engine_id": engine_id,
                "cycle": engine_df.iloc[i + window]["cycle"],
                "source_fd": engine_df.iloc[i + window]["source_fd"],
                "airline_id": engine_df.iloc[i + window]["airline_id"],
                "true_RUL": y[i + window],
                "true_fault": float(y[i + window] <= fault_threshold),
            })

    return (
        np.array(X_seq, dtype=np.float32),
        np.array(y_rul, dtype=np.float32),
        np.array(y_fault, dtype=np.float32),
        pd.DataFrame(meta)
    )

# Train-Test-Split Per Airline

In [8]:
client_data = {}

for airline_id, df in latent_airline_clients.items():

    engines = df["global_engine_id"].unique()

    train_engines, test_engines = train_test_split(
        engines,
        test_size=0.2,
        random_state=SEED,
        shuffle=True
    )

    train_df = df[
        df["global_engine_id"].isin(train_engines)
    ].copy()

    test_df = df[
        df["global_engine_id"].isin(test_engines)
    ].copy()

    X_train, y_train, f_train, train_meta = create_sequences(
        train_df,
        latent_features,
        WINDOW,
        FAULT_THRESHOLD
    )

    X_test, y_test, f_test, test_meta = create_sequences(
        test_df,
        latent_features,
        WINDOW,
        FAULT_THRESHOLD
    )

    client_data[airline_id] = {
        "X_train": X_train,
        "y_train": y_train,
        "f_train": f_train,
        "train_meta": train_meta,

        "X_test": X_test,
        "y_test": y_test,
        "f_test": f_test,
        "test_meta": test_meta,
    }

    print(
        airline_id,
        "train:",
        X_train.shape,
        "test:",
        X_test.shape
    )

Airline_1 train: (25633, 30, 64) test: (5835, 30, 64)
Airline_2 train: (29406, 30, 64) test: (6762, 30, 64)
Airline_3 train: (25512, 30, 64) test: (6728, 30, 64)
Airline_4 train: (30716, 30, 64) test: (8497, 30, 64)


# Dataloaders

In [9]:
loaders = {}

for airline_id, data in client_data.items():

    X_train = torch.tensor(
        data["X_train"],
        dtype=torch.float32
    )

    y_train = torch.tensor(
        data["y_train"],
        dtype=torch.float32
    )

    f_train = torch.tensor(
        data["f_train"],
        dtype=torch.float32
    )

    X_test = torch.tensor(
        data["X_test"],
        dtype=torch.float32
    )

    y_test = torch.tensor(
        data["y_test"],
        dtype=torch.float32
    )

    f_test = torch.tensor(
        data["f_test"],
        dtype=torch.float32
    )

    loaders[airline_id] = {
        "train": DataLoader(
            TensorDataset(X_train, y_train, f_train),
            batch_size=BATCH_SIZE,
            shuffle=True
        ),

        "test": DataLoader(
            TensorDataset(X_test, y_test, f_test),
            batch_size=BATCH_SIZE,
            shuffle=False
        )
    }

# Federated LSTM Model

In [10]:
class FL_LSTM(nn.Module):

    def __init__(self):

        super().__init__()

        self.lstm = nn.LSTM(
            input_size=64,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            dropout=0.2
        )

        self.rul_head = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

        self.fault_head = nn.Sequential(
            nn.Linear(128, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):

        out, _ = self.lstm(x)

        h = out[:, -1, :]

        rul = self.rul_head(h).squeeze(-1)

        fault_logit = self.fault_head(h).squeeze(-1)

        return rul, fault_logit

# Train and Evaluation Helper

In [11]:
def train_client(model, loader, epochs=1, lr=0.001):

    model.train()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr
    )

    mse = nn.MSELoss()
    bce = nn.BCEWithLogitsLoss()

    total_loss = 0.0
    total_batches = 0

    for epoch in range(epochs):

        for X, y, f in loader:

            X = X.to(DEVICE)
            y = y.to(DEVICE)
            f = f.to(DEVICE)

            optimizer.zero_grad()

            pred_rul, pred_fault_logit = model(X)

            loss_rul = mse(
                pred_rul,
                y
            )

            loss_fault = bce(
                pred_fault_logit,
                f
            )

            loss = loss_rul + loss_fault

            loss.backward()

            optimizer.step()

            total_loss += loss.item()
            total_batches += 1

    return (
        model.state_dict(),
        total_loss / max(total_batches, 1)
    )

In [12]:
def evaluate_model(model, loader):

    model.eval()

    all_true_rul = []
    all_pred_rul = []
    all_true_fault = []
    all_pred_fault_prob = []

    with torch.no_grad():

        for X, y, f in loader:

            X = X.to(DEVICE)

            pred_rul, pred_fault_logit = model(X)

            pred_fault_prob = torch.sigmoid(
                pred_fault_logit
            )

            all_true_rul.extend(
                y.numpy()
            )

            all_pred_rul.extend(
                pred_rul.cpu().numpy()
            )

            all_true_fault.extend(
                f.numpy()
            )

            all_pred_fault_prob.extend(
                pred_fault_prob.cpu().numpy()
            )

    all_true_rul = np.array(all_true_rul)
    all_pred_rul = np.array(all_pred_rul)
    all_true_fault = np.array(all_true_fault)
    all_pred_fault_prob = np.array(all_pred_fault_prob)

    pred_fault_label = (
        all_pred_fault_prob > 0.5
    ).astype(int)

    rmse = np.sqrt(
        np.mean(
            (np.array(all_true_rul) - np.array(all_pred_rul)) ** 2
        )
    )

    acc = accuracy_score(
        all_true_fault,
        pred_fault_label
    )

    auprc = average_precision_score(
        all_true_fault,
        all_pred_fault_prob
    )

    return {
        "rmse": rmse,
        "fault_accuracy": acc,
        "fault_auprc": auprc
    }

In [13]:
def predict_client(model, loader):

    model.eval()

    pred_rul_all = []
    pred_fault_prob_all = []
    pred_fault_label_all = []

    with torch.no_grad():

        for X, y, f in loader:

            X = X.to(DEVICE)

            pred_rul, pred_fault_logit = model(X)

            pred_fault_prob = torch.sigmoid(
                pred_fault_logit
            )

            pred_fault_label = (
                pred_fault_prob > 0.5
            ).int()

            pred_rul_all.extend(
                pred_rul.cpu().numpy()
            )

            pred_fault_prob_all.extend(
                pred_fault_prob.cpu().numpy()
            )

            pred_fault_label_all.extend(
                pred_fault_label.cpu().numpy()
            )

    return (
        np.array(pred_rul_all),
        np.array(pred_fault_prob_all),
        np.array(pred_fault_label_all)
    )

# Normal FedAvg Before Similarity

In [14]:
def fedavg(local_weights, client_sizes):

    total_size = sum(client_sizes)

    avg_weights = {}

    for key in local_weights[0].keys():

        avg_weights[key] = sum(
            local_weights[i][key].float()
            * client_sizes[i]
            / total_size
            for i in range(len(local_weights))
        )

    return avg_weights

# Predict

In [15]:
global_model = FL_LSTM().to(DEVICE)

ROUNDS = 10
LOCAL_EPOCHS = 1
LR = 0.001

fedavg_history = []


for round_num in range(ROUNDS):

    local_weights = []
    client_sizes = []
    round_losses = {}

    for airline_id in loaders:

        local_model = deepcopy(global_model).to(DEVICE)

        weights, loss = train_client(
            local_model,
            loaders[airline_id]["train"],
            epochs=LOCAL_EPOCHS,
            lr=LR
        )

        local_weights.append(weights)

        client_sizes.append(
            len(client_data[airline_id]["X_train"])
        )

        round_losses[airline_id] = loss

    global_model.load_state_dict(
        fedavg(local_weights, client_sizes)
    )

    row = {
        "round": round_num + 1
    }

    for airline_id, loss in round_losses.items():
        row[f"{airline_id}_loss"] = loss

    fedavg_history.append(row)

    print("FedAvg Round", round_num + 1, "completed")

FedAvg Round 1 completed
FedAvg Round 2 completed
FedAvg Round 3 completed
FedAvg Round 4 completed
FedAvg Round 5 completed
FedAvg Round 6 completed
FedAvg Round 7 completed
FedAvg Round 8 completed
FedAvg Round 9 completed
FedAvg Round 10 completed


# Save Predition

In [16]:
def save_method_predictions(models, method_name, file_prefix):

    all_predictions = []
    all_evaluations = []

    for airline_id in loaders:

        if isinstance(models, dict):
            model = models[airline_id]
        else:
            model = models

        pred_rul, pred_fault_prob, pred_fault_label = predict_client(
            model,
            loaders[airline_id]["test"]
        )

        meta = client_data[airline_id]["test_meta"].copy()

        meta["method"] = method_name
        meta["pred_RUL"] = pred_rul
        meta["pred_fault_prob"] = pred_fault_prob
        meta["pred_fault_label"] = pred_fault_label

        rmse = np.sqrt(
            mean_squared_error(
                meta["true_RUL"],
                meta["pred_RUL"]
            )
        )

        acc = accuracy_score(
            meta["true_fault"],
            meta["pred_fault_label"]
        )

        auprc = average_precision_score(
            meta["true_fault"],
            meta["pred_fault_prob"]
        )

        all_evaluations.append({
            "method": method_name,
            "airline_id": airline_id,
            "RUL_RMSE": rmse,
            "Fault_Accuracy": acc,
            "Fault_AUPRC": auprc
        })

        all_predictions.append(meta)

    prediction_df = pd.concat(
        all_predictions,
        ignore_index=True
    )

    evaluation_df = pd.DataFrame(
        all_evaluations
    )

    prediction_df.to_csv(
        f"{file_prefix}_predictions.csv",
        index=False
    )

    evaluation_df.to_csv(
        f"{file_prefix}_evaluation.csv",
        index=False
    )

    return prediction_df, evaluation_df

In [17]:
fedavg_predictions, fedavg_evaluation = save_method_predictions(
    global_model,
    method_name="FedAvg",
    file_prefix="fedavg"
)

print(fedavg_evaluation)

   method airline_id    RUL_RMSE  Fault_Accuracy  Fault_AUPRC
0  FedAvg  Airline_1  113.923837        0.704713     0.307224
1  FedAvg  Airline_2   90.721344        0.294883     0.204695
2  FedAvg  Airline_3   64.220350        0.773484     0.716358
3  FedAvg  Airline_4  144.992847        0.391785     0.185029


# MMD Dissimilarity Matrix

In [18]:
def rbf_kernel(X, Y, gamma=None):

    X = np.asarray(X, dtype=np.float32)
    Y = np.asarray(Y, dtype=np.float32)

    X_norm = np.sum(X ** 2, axis=1).reshape(-1, 1)
    Y_norm = np.sum(Y ** 2, axis=1).reshape(1, -1)

    dist = X_norm + Y_norm - 2 * np.dot(X, Y.T)

    if gamma is None:

        positive_dist = dist[dist > 0]

        if len(positive_dist) == 0:
            gamma = 1.0
        else:
            gamma = 1.0 / (
                2.0 * np.median(positive_dist) + 1e-8
            )

    return np.exp(-gamma * dist)

In [19]:
def mmd_rbf(X, Y, max_samples=1000):

    if len(X) > max_samples:
        X = X[
            np.random.choice(
                len(X),
                max_samples,
                replace=False
            )
        ]

    if len(Y) > max_samples:
        Y = Y[
            np.random.choice(
                len(Y),
                max_samples,
                replace=False
            )
        ]

    K_xx = rbf_kernel(X, X)
    K_yy = rbf_kernel(Y, Y)
    K_xy = rbf_kernel(X, Y)

    return (
        K_xx.mean()
        + K_yy.mean()
        - 2 * K_xy.mean()
    )

In [20]:
client_repr = {}

for airline_id in client_data:

    X_train = client_data[airline_id]["X_train"]

    # Each LSTM sample is window x 64.
    # Mean over window gives one 64-item vector.
    client_repr[airline_id] = X_train.mean(axis=1)

In [21]:
airline_ids = list(client_data.keys())

mmd_matrix = pd.DataFrame(
    index=airline_ids,
    columns=airline_ids,
    dtype=float
)

for a1 in airline_ids:

    for a2 in airline_ids:

        if a1 == a2:
            mmd_matrix.loc[a1, a2] = 0.0

        else:
            mmd_matrix.loc[a1, a2] = mmd_rbf(
                client_repr[a1],
                client_repr[a2]
            )

print(mmd_matrix)

mmd_matrix.to_csv(
    "mmd_dissimilarity_matrix.csv"
)

           Airline_1  Airline_2  Airline_3  Airline_4
Airline_1   0.000000  -0.226243  -0.275578  -0.208124
Airline_2  -0.224013   0.000000  -0.206824  -0.110584
Airline_3  -0.277355  -0.181764   0.000000  -0.137649
Airline_4  -0.215484  -0.112624  -0.167642   0.000000


# Convert MMD to Similarity

In [22]:
mmd_values = mmd_matrix.values

positive_mmd = mmd_values[
    mmd_values > -0.5
]

tau = positive_mmd.mean()

similarity_matrix = np.exp(
    -mmd_matrix / tau
)

similarity_matrix = pd.DataFrame(
    similarity_matrix,
    index=mmd_matrix.index,
    columns=mmd_matrix.columns
)

print(similarity_matrix)

similarity_matrix.to_csv(
    "mmd_similarity_matrix.csv"
)

           Airline_1  Airline_2  Airline_3  Airline_4
Airline_1   1.000000   0.213440   0.152411   0.241542
Airline_2   0.216714   1.000000   0.243695   0.470069
Airline_3   0.150574   0.289161   1.000000   0.390773
Airline_4   0.229706   0.463569   0.318424   1.000000


# Similarity Aware FedAvg

In [23]:
def similarity_weighted_fedavg(
    local_weights,
    similarity_scores,
    client_sizes
):

    raw_weights = (
        np.array(similarity_scores, dtype=np.float32)
        * np.array(client_sizes, dtype=np.float32)
    )

    raw_weights = raw_weights / raw_weights.sum()

    avg_weights = {}

    for key in local_weights[0].keys():

        avg_weights[key] = sum(
            local_weights[i][key].float()
            * raw_weights[i]
            for i in range(len(local_weights))
        )

    return avg_weights

In [24]:
personalized_models = {
    airline_id: FL_LSTM().to(DEVICE)
    for airline_id in airline_ids
}

similarity_history = []

ROUNDS = 10
LOCAL_EPOCHS = 1
LR = 0.001


for round_num in range(ROUNDS):

    local_weights_dict = {}
    client_sizes_dict = {}
    round_losses = {}

    for airline_id in airline_ids:

        local_model = deepcopy(
            personalized_models[airline_id]
        ).to(DEVICE)

        weights, loss = train_client(
            local_model,
            loaders[airline_id]["train"],
            epochs=LOCAL_EPOCHS,
            lr=LR
        )

        local_weights_dict[airline_id] = weights

        client_sizes_dict[airline_id] = len(
            client_data[airline_id]["X_train"]
        )

        round_losses[airline_id] = loss

    for target_airline in airline_ids:

        source_weights = [
            local_weights_dict[source_airline]
            for source_airline in airline_ids
        ]

        similarity_scores = [
            similarity_matrix.loc[
                target_airline,
                source_airline
            ]
            for source_airline in airline_ids
        ]

        source_sizes = [
            client_sizes_dict[source_airline]
            for source_airline in airline_ids
        ]

        new_weights = similarity_weighted_fedavg(
            source_weights,
            similarity_scores,
            source_sizes
        )

        personalized_models[target_airline].load_state_dict(
            new_weights
        )

    row = {
        "round": round_num + 1
    }

    for airline_id, loss in round_losses.items():
        row[f"{airline_id}_loss"] = loss

    similarity_history.append(row)

    print(
        "Similarity-aware FL Round",
        round_num + 1,
        "completed"
    )

Similarity-aware FL Round 1 completed
Similarity-aware FL Round 2 completed
Similarity-aware FL Round 3 completed
Similarity-aware FL Round 4 completed
Similarity-aware FL Round 5 completed
Similarity-aware FL Round 6 completed
Similarity-aware FL Round 7 completed
Similarity-aware FL Round 8 completed
Similarity-aware FL Round 9 completed
Similarity-aware FL Round 10 completed


# Save Similarity Aware Prediction

In [25]:
sim_predictions, sim_evaluation = save_method_predictions(
    personalized_models,
    method_name="Similarity_Aware_FL",
    file_prefix="similarity_aware_fl"
)

print(sim_evaluation)

                method airline_id    RUL_RMSE  Fault_Accuracy  Fault_AUPRC
0  Similarity_Aware_FL  Airline_1   67.460242        0.711568     0.307137
1  Similarity_Aware_FL  Airline_2   63.786685        0.705856     0.275369
2  Similarity_Aware_FL  Airline_3   71.538789        0.749851     0.259835
3  Similarity_Aware_FL  Airline_4  103.600094        0.765917     0.200931


# Compare Normal FL vs Similarity FL

In [26]:
comparison_df = pd.concat(
    [
        fedavg_evaluation,
        sim_evaluation
    ],
    ignore_index=True
)

comparison_df.to_csv(
    "fedavg_vs_similarity_aware_fl.csv",
    index=False
)

print(comparison_df)

                method airline_id    RUL_RMSE  Fault_Accuracy  Fault_AUPRC
0               FedAvg  Airline_1  113.923837        0.704713     0.307224
1               FedAvg  Airline_2   90.721344        0.294883     0.204695
2               FedAvg  Airline_3   64.220350        0.773484     0.716358
3               FedAvg  Airline_4  144.992847        0.391785     0.185029
4  Similarity_Aware_FL  Airline_1   67.460242        0.711568     0.307137
5  Similarity_Aware_FL  Airline_2   63.786685        0.705856     0.275369
6  Similarity_Aware_FL  Airline_3   71.538789        0.749851     0.259835
7  Similarity_Aware_FL  Airline_4  103.600094        0.765917     0.200931


In [27]:
summary_df = comparison_df.groupby("method")[
    [
        "RUL_RMSE",
        "Fault_Accuracy",
        "Fault_AUPRC"
    ]
].mean()

print(summary_df)

                       RUL_RMSE  Fault_Accuracy  Fault_AUPRC
method                                                      
FedAvg               103.464595        0.541216     0.353326
Similarity_Aware_FL   76.596453        0.733298     0.260818
